Contexto de lo que tengo en azureML:
tengo el dataset ya:
    my_training_data_input = Input(
        type=AssetTypes.MLTABLE,
        path="azureml:diabetes-dataset:1"
    )
tengo un recurso de compute ya creado:
    compute_target = "cpu-cluster"

In [1]:
import os
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.constants import AssetTypes


mlclient = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=os.getenv("AZURE_SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("AZURE_RESOURCE_GROUP"),
    workspace_name=os.getenv("AZURE_WORKSPACE_NAME")
)


PARA CREAR UN JOB con 

In [2]:
import mlflow
import azure.ai.ml.automl as automl
from azure.ai.ml import Input


# Crear la referencia de entrada al dataset registrado en Azure
my_training_data_input = Input(
    type=AssetTypes.MLTABLE,
    path="azureml:diabetes-dataset:1"
)

# Configurar la tarea de clasificación
# auro vamos a crear un experimento de AutoML para clasificación
# automl buscara el mejor modelo de clasificación para predecir la columna "target" del dataset
job = automl.classification(
    experiment_name="diabetes-automl_future_pipeline",
    training_data=my_training_data_input,  # ← Usamos el Input, NO el objeto Data
    target_column_name="target",
    compute="cpu-cluster",
    primary_metric="accuracy"
    # featurization="auto",  
    # "auto" los datos se preprocesan (normalizan, escala, imputan) automáticamente (por defecto)
    # "off" ← los datos no se modifican
    # "custom" ← se puede definir un diccionario de preprocesamiento personalizado (ver documentación AzureML)
)

job.set_limits(
    timeout_minutes=60,
    trial_timeout_minutes=20,
    max_trials=5,
    enable_early_termination=True,
)

# Lanzar el experimento
returned_job = mlclient.jobs.create_or_update(job)
print(f"Job AutoML iniciado: {returned_job.name}")
print(f"Sigue el progreso en Azure ML Studio → Jobs → {returned_job.name}")
print(f"Estado del job: {returned_job.status}")

/home/renato/Desktop/MLOps_GHActions_practicas/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Job AutoML iniciado: heroic_truck_zh5ff61ydf
Sigue el progreso en Azure ML Studio → Jobs → heroic_truck_zh5ff61ydf
Estado del job: NotStarted


---
## **Fase 5: Entrenamiento por comandos (YAML)**

### 5.1 Ejecutar job desde `job.yml`

Lanzamos el entrenamiento directamente desde el archivo YAML configurado con `LogisticRegression`.

In [ ]:
from azure.ai.ml import load_job

# Cargar job desde el archivo YAML
job_from_yml = load_job(source="./src/job.yml")

# Lanzar el job
returned_job_yml = mlclient.jobs.create_or_update(job_from_yml)
print(f"Job desde YAML iniciado: {returned_job_yml.name}")
print(f"Ver en Azure ML Studio → Jobs → {returned_job_yml.name}")

ResourceNotFoundError: (UserError) No environment exists for name: AzureML-sklearn-1.0-ubuntu20.04-py38-cpu, version: 
Code: UserError
Message: No environment exists for name: AzureML-sklearn-1.0-ubuntu20.04-py38-cpu, version: 